In [1]:
#libraries
import pandas as pd
import numpy as np
import os
import sys
from glob import glob
import joblib
import warnings
from datetime import date, datetime
import copy

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from scipy.stats import pearsonr
import scipy.stats as st

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

from scipy.stats import zscore as  zscore
import pickle


#from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import TransformedTargetRegressor


In [2]:
#path to dir
path='/scratch2/alinat/project/PD-EEG-ANON_eegOnly/MLtables180'

In [3]:
#load tables
feat_tables = {}
files = sorted(glob(path+'/*.csv'))
for file in files:
    if 'COG_' not in file:
        if 'ad-hoc' not in file:
            nm = file.split('/')[-1].split('.csv')[0]
            feat_tables[nm] = pd.read_csv(file, index_col=0)
        

targ_table = pd.read_csv(sorted(glob(path+'/*COG_*'))[0], index_col=0)


In [7]:
#split to folds
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)


target = pd.DataFrame(targ_table['global_z_no_language'])

data_indices = target.index
#make folds
Folds = {}
for fold, (train_indices, test_indices) in enumerate(kf.split(target.index), 0):
    #get indexes
    train_sessions = target.index[train_indices]
    test_sessions = target.index[test_indices]
    #make fold object
    Folds['fold_'+str(fold)] = {}

    for tset_id, tset_nm in zip([train_sessions, test_sessions],
                                ['Train', 'Test']): 
        #store id, target and features for train and test
        Folds['fold_'+str(fold)][tset_nm] = {}
        Folds['fold_'+str(fold)][tset_nm]['ids'] = tset_id
        Folds['fold_'+str(fold)][tset_nm]['target'] = target.reindex(index=tset_id)
        Folds['fold_'+str(fold)][tset_nm]['features'] = {}
        for key in feat_tables.keys():
            Folds['fold_'+str(fold)][tset_nm]['features'][key] = feat_tables[key].reindex(index=tset_id)



   

In [8]:
#save
fnm = '/MLout/DCT_1targs_anat_rest_4fold_ZnoLG.pkl'
# Save to file
with open(path+fnm, 'wb') as file:
    pickle.dump(Folds, file)

# # Load from file
# with open(path+fnm, 'rb') as file:
#     Folds = pickle.load(file)